### Data Reading

In [ ]:
df1=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/tutorial_learning/raw/customers.csv")

In [ ]:
df1.display()

customer_id,customer_name,city,state,country,signup_date
1,Ritu Ghosh,Howrah,West Bengal,India,2022-06-26
2,Aryan Bose,Bhopal,Madhya Pradesh,India,2022-12-07
3,Deepa Tiwari,Indore,Madhya Pradesh,India,2022-02-26
4,Amit Tiwari,Warangal,Telangana,India,2024-05-04
5,Rajesh Verma,Kolkata,West Bengal,India,2024-05-01
6,Aditya Joshi,Surat,Gujarat,India,2023-03-24
7,Vikram Rao,Warangal,Telangana,India,2022-07-22
8,Simran Menon,Guwahati,Assam,India,2023-11-28
9,Vivaan Roy,Ranchi,Jharkhand,India,2023-09-25
10,Gaurav Pandey,Bhopal,Madhya Pradesh,India,2024-01-08


In [ ]:
df2=spark.read.format("csv").option("header","true").option("inferSchema","true").load("/Volumes/workspace/default/tutorial_learning/raw/orders.csv")
df2.display()

order_id,customer_id,order_date,payment_method,order_status
1,31,2025-11-30,Credit Card,Completed
2,1893,2024-10-27,UPI,Completed
3,1083,2024-04-20,Net Banking,Returned
4,1916,2024-11-23,UPI,Completed
5,898,2022-09-13,UPI,Completed
6,1377,2025-05-24,UPI,Completed
7,935,2024-02-13,UPI,Completed
8,445,2025-08-19,UPI,Completed
9,1259,2024-01-17,Credit Card,Cancelled
10,1131,2024-08-22,Cash on Delivery,Processing


In [ ]:
df3=spark.read.format("csv").option("header",True).option("inferSchema",True).load("/Volumes/workspace/default/tutorial_learning/raw/order_items.csv")
df3.display()

order_id,product_id,quantity,unit_price
1,35,1,2865.41
2,29,1,4319.50
2,13,3,2131.33
3,18,1,1854.55
4,60,5,579.94
5,33,1,312.93
6,78,1,876.32
6,72,1,1918.28
6,26,1,1210.27
7,92,1,1591.57


In [ ]:
df4=spark.read.format("csv").option("header",True).option("inferSchema",True).load("/Volumes/workspace/default/tutorial_learning/raw/products.csv")
df4.display()

product_id,product_name,category,price
1,Smartphone Pro Max,Electronics,28043.24
2,Wireless Noise Cancelling Earbuds,Electronics,3011.06
3,Ultra-Slim 14-inch Laptop,Electronics,64028.30
4,Fitness Smartwatch with SpO2,Electronics,2324.92
5,Portable Bluetooth Speaker 20W,Electronics,3026.39
6,43-inch 4K Ultra HD Smart TV,Electronics,22709.90
7,Ergonomic Optical Gaming Mouse,Electronics,1169.34
8,Mechanical RGB Gaming Keyboard,Electronics,2448.97
9,10-inch Android Tablet,Electronics,11424.27
10,20000mAh Fast Charging Power Bank,Electronics,1936.14


In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

### Order value rollup

In [ ]:
df_ord_price=df3.groupBy('order_id').agg(round(sum(col('unit_price')*col('quantity')),0).alias("total_price"))
df_ord_price.display()

order_id,total_price
12,6090.0
18,6794.0
38,367.0
67,3641.0
70,5687.0
93,33742.0
161,7537.0
186,558.0
190,471.0
218,27889.0


In [ ]:
df_ord_price.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- total_price: double (nullable = true)



###  Validate data types 

In [ ]:
df1.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- signup_date: date (nullable = true)



In [ ]:
df2.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [ ]:
df3.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)



In [ ]:
df4.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price: double (nullable = true)



### Check duplicates in all four tables

In [ ]:
df1.createOrReplaceTempView("customers")
df2.createOrReplaceTempView("orders")
df3.createOrReplaceTempView("order_items")
df4.createOrReplaceTempView("products")
df_ord_price.createOrReplaceTempView("order_price")

In [ ]:
print("customers",df1.count())
print("orders",df2.count())
print("order_items",df3.count())
print("products",df4.count())

customers 2015
orders 10000
order_items 21968
products 100


In [ ]:
print("customers",df1.distinct().count())
print("orders",df2.distinct().count())
print("order_items",df3.distinct().count())
print("products",df4.distinct().count())

customers 2000
orders 10000
order_items 21968
products 100


#### 15 duplicate records in customers table

In [ ]:
select customer_id,count(customer_id) from customers group by customer_id  having count(customer_id)>1

customer_id,count(customer_id)
1111,2
1629,2
376,2
1040,2
1261,2
1433,2
460,2
374,2
1163,2
48,2


### Joining all tables to get complete info

In [ ]:
select 
  c.customer_id,
  c.customer_name,
  c.city,
  c.state,
  c.country,
  c.signup_date,
  o.order_id,
  o.order_date,
  o.payment_method,
  o.order_status,
  i.product_id,
  i.quantity,
  i.unit_price,
  p.product_name,
  p.category,
  p.price
from customers as c
left join orders as o on c.customer_id = o.customer_id
left join order_items as i on o.order_id = i.order_id
left join products as p on i.product_id = p.product_id

customer_id,customer_name,city,state,country,signup_date,order_id,order_date,payment_method,order_status,product_id,quantity,unit_price,product_name,category,price
1,Ritu Ghosh,Howrah,West Bengal,India,2022-06-26,8851,2024-05-17,UPI,Completed,50,1,1105.68,Silent Sweep Quartz Wall Clock 12-inch,Home & Kitchen,1105.68
2,Aryan Bose,Bhopal,Madhya Pradesh,India,2022-12-07,4726,2025-08-02,Debit Card,Completed,90,3,492.26,Caffeine Infused Under Eye Cream 20g,Beauty,492.26
3,Deepa Tiwari,Indore,Madhya Pradesh,India,2022-02-26,3764,2022-05-07,Credit Card,Shipped,45,2,1626.32,100% Pure Cotton King Size Bedsheet,Home & Kitchen,1626.32
4,Amit Tiwari,Warangal,Telangana,India,2024-05-04,9209,2025-02-13,Credit Card,Completed,96,1,320.77,Roasted Granulated Instant Coffee 100g,Grocery,320.77
5,Rajesh Verma,Kolkata,West Bengal,India,2024-05-01,6356,2025-04-01,UPI,Cancelled,45,1,1626.32,100% Pure Cotton King Size Bedsheet,Home & Kitchen,1626.32
6,Aditya Joshi,Surat,Gujarat,India,2023-03-24,6502,2025-02-13,UPI,Completed,34,1,709.99,Athletic Training Track Pants,Clothing,709.99
7,Vikram Rao,Warangal,Telangana,India,2022-07-22,8605,2024-09-26,UPI,Completed,99,1,458.92,Creamy Unsweetened Peanut Butter 1kg,Grocery,458.92
8,Simran Menon,Guwahati,Assam,India,2023-11-28,6430,2025-02-07,Credit Card,Completed,36,4,601.90,Soft Merino Woolen Muffler Scarf,Clothing,601.90
9,Vivaan Roy,Ranchi,Jharkhand,India,2023-09-25,9378,2025-07-06,Debit Card,Shipped,48,1,1218.45,Airtight Food Storage Containers (Set of 6),Home & Kitchen,1218.45
10,Gaurav Pandey,Bhopal,Madhya Pradesh,India,2024-01-08,8613,2024-05-18,UPI,Returned,53,2,620.53,Silicone Non-Stick Baking Mat,Home & Kitchen,620.53


### Phase 1: Order and sales analysis

#### Order-level dataset (orders + order value)

In [ ]:
select o.*,op.total_price from orders as o left join order_price as op on o.order_id = op.order_id

order_id,customer_id,order_date,payment_method,order_status,total_price
1,31,2025-11-30,Credit Card,Completed,2865.0
2,1893,2024-10-27,UPI,Completed,10713.0
3,1083,2024-04-20,Net Banking,Returned,1855.0
4,1916,2024-11-23,UPI,Completed,2900.0
5,898,2022-09-13,UPI,Completed,313.0
6,1377,2025-05-24,UPI,Completed,4005.0
7,935,2024-02-13,UPI,Completed,4749.0
8,445,2025-08-19,UPI,Completed,5557.0
9,1259,2024-01-17,Credit Card,Cancelled,2391.0
10,1131,2024-08-22,Cash on Delivery,Processing,5113.0


In [ ]:
dfo1=dfo.withColumn('month',month('order_date')).withColumn('year',year('order_date'))

In [ ]:
dfo1.display()

order_id,customer_id,order_date,payment_method,order_status,total_price,month,year
1,31,2025-11-30,Credit Card,Completed,2865.0,11,2025
2,1893,2024-10-27,UPI,Completed,10713.0,10,2024
3,1083,2024-04-20,Net Banking,Returned,1855.0,4,2024
4,1916,2024-11-23,UPI,Completed,2900.0,11,2024
5,898,2022-09-13,UPI,Completed,313.0,9,2022
6,1377,2025-05-24,UPI,Completed,4005.0,5,2025
7,935,2024-02-13,UPI,Completed,4749.0,2,2024
8,445,2025-08-19,UPI,Completed,5557.0,8,2025
9,1259,2024-01-17,Credit Card,Cancelled,2391.0,1,2024
10,1131,2024-08-22,Cash on Delivery,Processing,5113.0,8,2024


#### Monthly Revenue

In [ ]:
df_monthly=dfo1.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('year','month').agg(countDistinct('order_id').alias('total_orders'),sum('total_price').alias('total_revenue')).orderBy('year','month')

In [ ]:
df_monthly.display()

year,month,total_orders,total_revenue
2022,1,2,4866.0
2022,2,6,82008.0
2022,3,13,192515.0
2022,4,31,293577.0
2022,5,30,266221.0
2022,6,33,278930.0
2022,7,39,478624.0
2022,8,44,418824.0
2022,9,50,456042.0
2022,10,58,589618.0


#### Yearly Revenue

In [ ]:
df_yearly=dfo1.filter(col('year')<=2025).filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('year').agg(sum('total_price').alias('total_revenue'),countDistinct('order_id').alias('total_orders')).orderBy('year')

In [ ]:
df_yearly.withColumn('AOV',round(col('total_revenue')/col('total_orders'),2)).display()

year,total_revenue,total_orders,AOV
2022,4275502.0,440,9717.05
2023,15913586.0,1564,10174.93
2024,32496390.0,3162,10277.16
2025,39026588.0,3363,11604.69


#### Order Status Analysis

In [ ]:
df.groupby('order_status').agg(count('customer_id').alias('count')).display()

order_status,count
Shipped,3448
Returned,1047
None,15
Cancelled,1562
Processing,1555
Completed,14407


#### Payment method analysis

In [ ]:
dfo1.groupby('payment_method').agg(countDistinct('order_id').alias('total_orders'),sum('total_price').alias('total_revenue')).display()

payment_method,total_orders,total_revenue
Credit Card,2129,23894333.0
Debit Card,1452,16808488.0
UPI,4346,45587980.0
Cash on Delivery,1202,10949197.0
None,60,567481.0
Net Banking,811,9248320.0


### Phase 2: Product analysis

In [ ]:
select oi.*,o.order_status,p.product_name,p.category,p.price from order_items as oi left join orders as o on oi.order_id = o.order_id left join products as p on oi.product_id = p.product_id

order_id,product_id,quantity,unit_price,order_status,product_name,category,price
1,35,1,2865.41,Completed,Pure Linen Casual Button-Down Shirt,Clothing,2865.41
2,29,1,4319.50,Completed,Pure Banarasi Art Silk Saree,Clothing,4319.50
2,13,3,2131.33,Completed,7-in-1 USB-C Multiport Adapter,Electronics,2131.33
3,18,1,1854.55,Returned,Wireless Charging Pad 15W,Electronics,1854.55
4,60,5,579.94,Completed,Sapiens: A Brief History of Humankind,Books,579.94
5,33,1,312.93,Completed,Cotton Ankle Length Socks (Pack of 3),Clothing,312.93
6,78,1,876.32,Completed,Anti-Fog UV Protection Swimming Goggles,Sports,876.32
6,72,1,1918.28,Completed,Kashmir Willow Full Size Cricket Bat,Sports,1918.28
6,26,1,1210.27,Completed,Women's Casual Cotton Maxi Dress,Clothing,1210.27
7,92,1,1591.57,Completed,Cold-Pressed Extra Virgin Olive Oil 1L,Grocery,1591.57


#### Top 10 products by revenue

In [ ]:
dfo2.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('product_name').agg(sum('quantity').alias('total_quantity'),round(sum(col('quantity')*col('unit_price')),0).alias('total_revenue')).orderBy(col('total_revenue').desc()).limit(10).display()

product_name,total_quantity,total_revenue
Ultra-Slim 14-inch Laptop,320,20489056.0
Smartphone Pro Max,328,9198183.0
43-inch 4K Ultra HD Smart TV,352,7993885.0
Mini Portable LED Projector,377,5071016.0
10-inch Android Tablet,401,4581132.0
Over-Ear Studio Headphones,314,2970917.0
Digital Air Fryer 4.5L,423,2761716.0
Action Camera 4K Waterproof,323,2497294.0
1TB External Solid State Drive,347,1974690.0
Pure Banarasi Art Silk Saree,344,1485908.0


#### Category performance

In [ ]:
from pyspark.sql.window import Window

In [ ]:
dfo2.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('category').agg(sum('quantity').alias('total_quantity'),countDistinct('order_id').alias('total_orders'),round(sum(col('quantity')*col('unit_price')),0).alias('total_revenue')).withColumn('ranking',rank().over(Window.partitionBy().orderBy(col('total_revenue').desc()))).display()

category,total_quantity,total_orders,total_revenue,ranking
Electronics,6719,3262,63647350.0,1
Clothing,5995,2960,9763146.0,2
Home & Kitchen,5497,2730,9338323.0,3
Sports,3858,2028,5025283.0,4
Books,4712,2413,2863605.0,5
Grocery,3367,1742,1847607.0,6
Beauty,3402,1802,1803581.0,7
None,189,104,410505.0,8


### Phase 3: Customer analysis

#### Overview of the joined data

In [ ]:
df.display()

customer_id,customer_name,city,state,country,signup_date,order_id,order_date,payment_method,order_status,product_id,quantity,unit_price,product_name,category,price
1,Ritu Ghosh,Howrah,West Bengal,India,2022-06-26,8851,2024-05-17,UPI,Completed,50,1,1105.68,Silent Sweep Quartz Wall Clock 12-inch,Home & Kitchen,1105.68
2,Aryan Bose,Bhopal,Madhya Pradesh,India,2022-12-07,4726,2025-08-02,Debit Card,Completed,90,3,492.26,Caffeine Infused Under Eye Cream 20g,Beauty,492.26
3,Deepa Tiwari,Indore,Madhya Pradesh,India,2022-02-26,3764,2022-05-07,Credit Card,Shipped,45,2,1626.32,100% Pure Cotton King Size Bedsheet,Home & Kitchen,1626.32
4,Amit Tiwari,Warangal,Telangana,India,2024-05-04,9209,2025-02-13,Credit Card,Completed,96,1,320.77,Roasted Granulated Instant Coffee 100g,Grocery,320.77
5,Rajesh Verma,Kolkata,West Bengal,India,2024-05-01,6356,2025-04-01,UPI,Cancelled,45,1,1626.32,100% Pure Cotton King Size Bedsheet,Home & Kitchen,1626.32
6,Aditya Joshi,Surat,Gujarat,India,2023-03-24,6502,2025-02-13,UPI,Completed,34,1,709.99,Athletic Training Track Pants,Clothing,709.99
7,Vikram Rao,Warangal,Telangana,India,2022-07-22,8605,2024-09-26,UPI,Completed,99,1,458.92,Creamy Unsweetened Peanut Butter 1kg,Grocery,458.92
8,Simran Menon,Guwahati,Assam,India,2023-11-28,6430,2025-02-07,Credit Card,Completed,36,4,601.90,Soft Merino Woolen Muffler Scarf,Clothing,601.90
9,Vivaan Roy,Ranchi,Jharkhand,India,2023-09-25,9378,2025-07-06,Debit Card,Shipped,48,1,1218.45,Airtight Food Storage Containers (Set of 6),Home & Kitchen,1218.45
10,Gaurav Pandey,Bhopal,Madhya Pradesh,India,2024-01-08,8613,2024-05-18,UPI,Returned,53,2,620.53,Silicone Non-Stick Baking Mat,Home & Kitchen,620.53


#### Top customers by spending

In [ ]:
df.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('customer_id','customer_name').agg(countDistinct('order_id').alias('total_orders'),round(sum((col('quantity')*col('unit_price'))),2).alias('total_spent')).orderBy(col('total_spent').desc()).limit(10).display()

customer_id,customer_name,total_orders,total_spent
210,Ananya Roy,4,428998.00
568,Sanjay Dutta,7,424528.38
855,Divya Kumar,7,374729.07
288,Jyoti Pillai,8,372079.79
1144,Swati Yadav,10,367364.30
504,Aryan Bose,7,351973.60
1757,Prisha Patil,4,350832.32
1425,Shruti Choudhury,4,346743.27
148,Ritu Nambiar,5,346112.89
1784,Ritu Iyer,4,345919.10


#### Sales by state or city

In [ ]:
df.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('state').agg(countDistinct('order_id').alias('total_orders'),sum('quantity').alias('total_quantity'),round(sum((col('quantity')*col('unit_price'))),0).alias('total_revenue')).orderBy(col('total_revenue').desc()).display()

state,total_orders,total_quantity,total_revenue
Gujarat,995,3876,11070615.0
Uttar Pradesh,877,3417,9871382.0
Maharashtra,791,3047,8791865.0
Punjab,625,2491,7627816.0
West Bengal,618,2427,7138737.0
Madhya Pradesh,667,2449,6948088.0
Kerala,646,2609,6899619.0
Rajasthan,608,2300,6476100.0
Telangana,555,2147,5304835.0
Karnataka,488,1863,5036336.0


In [ ]:
df.filter(col('order_status').isin(['Shipped','Processing','Completed'])).groupby('city').agg(countDistinct('order_id').alias('total_orders'),sum('quantity').alias('total_quantity'),round(sum((col('quantity')*col('unit_price'))),0).alias('total_revenue')).orderBy(col('total_revenue').desc()).display()

city,total_orders,total_quantity,total_revenue
Ludhiana,372,1481,4522516.0
Indore,387,1418,4224814.0
Noida,303,1224,3996264.0
Vadodara,333,1280,3949813.0
Kolkata,301,1185,3903150.0
Ranchi,263,1008,3766503.0
Kochi,330,1336,3727725.0
Ahmedabad,385,1495,3699889.0
Surat,270,1070,3361049.0
Jaipur,325,1224,3352933.0


### Phase 4: Advanced analysis

#### Cancelled orders and potential lost sales

In [ ]:
df.filter(col('order_status').isin(['Cancelled','Completed'])).groupby('order_status').agg(countDistinct('order_id').alias('total_orders'),round(sum((col('quantity')*col('unit_price'))),0).alias('total_revenue')).display()

order_status,total_orders,total_revenue
Cancelled,686,7477568.0
Completed,6520,69957542.0


#### Customer signup cohort

In [ ]:
select c.*,order_id,o.order_date,o.payment_method,o.order_status from customers as c left join orders as o on c.customer_id = o.customer_id

customer_id,customer_name,city,state,country,signup_date,order_id,order_date,payment_method,order_status,signup_month,signupyear
1,Ritu Ghosh,Howrah,West Bengal,India,2022-06-26,8851,2024-05-17,UPI,Completed,6,2022
2,Aryan Bose,Bhopal,Madhya Pradesh,India,2022-12-07,4726,2025-08-02,Debit Card,Completed,12,2022
3,Deepa Tiwari,Indore,Madhya Pradesh,India,2022-02-26,3764,2022-05-07,Credit Card,Shipped,2,2022
4,Amit Tiwari,Warangal,Telangana,India,2024-05-04,9209,2025-02-13,Credit Card,Completed,5,2024
5,Rajesh Verma,Kolkata,West Bengal,India,2024-05-01,6356,2025-04-01,UPI,Cancelled,5,2024
6,Aditya Joshi,Surat,Gujarat,India,2023-03-24,6502,2025-02-13,UPI,Completed,3,2023
7,Vikram Rao,Warangal,Telangana,India,2022-07-22,8605,2024-09-26,UPI,Completed,7,2022
8,Simran Menon,Guwahati,Assam,India,2023-11-28,6430,2025-02-07,Credit Card,Completed,11,2023
9,Vivaan Roy,Ranchi,Jharkhand,India,2023-09-25,9378,2025-07-06,Debit Card,Shipped,9,2023
10,Gaurav Pandey,Bhopal,Madhya Pradesh,India,2024-01-08,8613,2024-05-18,UPI,Returned,1,2024


In [ ]:
df_info=df_info.withColumn('signup_month',month('signup_date')).withColumn('signupyear',year('signup_date'))

In [ ]:
df_info.groupby('signupyear','signup_month').agg(countDistinct('customer_id').alias('total_customers')).orderBy('signupyear','signup_month').display()

signupyear,signup_month,total_customers
2022,1,65
2022,2,61
2022,3,85
2022,4,54
2022,5,74
2022,6,64
2022,7,69
2022,8,57
2022,9,64
2022,10,63


In [ ]:
df_info.groupby('signupyear', 'signup_month').agg(
    countDistinct('customer_id').alias('total_customers'),
    countDistinct(when(col('order_id').isNotNull(), col('customer_id'))).alias('customers_who_ordered')
).withColumn(
    'conversion_rate',
    round(col('customers_who_ordered') / col('total_customers') * 100, 2)
).orderBy('signupyear', 'signup_month').display()

signupyear,signup_month,total_customers,customers_who_ordered,conversion_rate
2022,1,65,65,100.00
2022,2,61,61,100.00
2022,3,85,85,100.00
2022,4,54,54,100.00
2022,5,74,73,98.65
2022,6,64,63,98.44
2022,7,69,68,98.55
2022,8,57,56,98.25
2022,9,64,63,98.44
2022,10,63,62,98.41


# Ecommerce Sales Analytics -- PySpark ETL Project

> End-to-end ETL + analytics pipeline on Databricks: raw e-commerce CSVs to validated & joined DataFrames to four phases of business analysis.

---

## Dataset

Raw CSVs loaded from Unity Catalog volume workspace.default.tutorial_learning.

| File | Rows | Key Columns |
|---|---:|---|
| customers.csv | 2,015 (2,000 distinct) | customer_id, customer_name, city, state, country, signup_date |
| orders.csv | 10,000 | order_id, customer_id, order_date, payment_method, order_status |
| order_items.csv | 21,968 | order_id, product_id, quantity, unit_price |
| products.csv | 100 | product_id, product_name, category, price |

## Key Insights

* ~9x revenue growth from 2022 ($4.3M) to 2025 ($39.0M)
* Electronics dominates -- $63.6M (~67% of product revenue)
* UPI is the #1 payment method -- 3,823 orders, $40.7M revenue
* ~$12.4M potential lost sales (cancelled + returned orders)
* 99.4% customer conversion rate
* Gujarat is the top state ($11.1M revenue)

## Tech Stack

* Databricks Serverless compute
* PySpark DataFrame API
* Spark SQL with temp views
* Window functions (rank())
* Unity Catalog Volumes for raw-file storage

## How to Run

1. Attach a Serverless compute to this notebook.
2. Run all cells top-to-bottom.
3. Requires read access to /Volumes/workspace/default/tutorial_learning/raw/.